# Cohort Retention Analysis

Measure customer retention over time using monthly cohorts built from each customer's first observed purchase. The notebook compares retention by signup month, market source proxy, and customer type proxy, then translates those differences into concrete product and marketing implications.


## What This Notebook Answers

- How does retention evolve by signup month cohort?
- Which cohorts retain best after 1, 3, and 6 months?
- Do domestic and international cohorts behave differently?
- Do higher-value first-time customers retain better than lower-value ones?
- What do cohort differences imply for activation, onboarding, and acquisition strategy?


## Dataset And Cohort Definitions

**Dataset:** `Online Retail.xlsx` from the local workspace.

Core fields used:

- `CustomerID`: customer identifier
- `InvoiceDate`: transaction timestamp
- `InvoiceNo`: order identifier
- `Country`: market / source proxy
- `Quantity` and `UnitPrice`: used to estimate basket value

Definitions used in this notebook:

- **Signup month cohort:** the month of a customer's first observed purchase
- **Retention:** a customer is retained in month $n$ if they place at least one order in cohort month $n$
- **Source proxy:** `Domestic (UK)` vs `International`, using `Country` because true acquisition channel is not present
- **Customer type proxy:** `Entry`, `Core`, and `Premium`, based on first-order value terciles

This is a purchase-retention view, not account-login retention. The strategy implications should therefore focus on repeat purchasing and reactivation.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 5)

DATA_PATH_CANDIDATES = [
    Path("../CLV Online Retail/Online Retail.xlsx"),
    Path("../CLV Non-Contractual/Online Retail.xlsx"),
    Path("../../Classification/Customer Segmentation - E-Commerce/data/online_retail_II.xlsx"),
]


def resolve_path(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find Online Retail workbook from: {candidates}")


DATA_PATH = resolve_path(DATA_PATH_CANDIDATES)
print(f"Using data file: {DATA_PATH}")


Using data file: ..\CLV Online Retail\Online Retail.xlsx


## Load The Transaction Data

The Online Retail workbook is large, so load only the columns needed for retention analysis and first-order cohort segmentation.


In [2]:
usecols = ["InvoiceNo", "InvoiceDate", "CustomerID", "Quantity", "UnitPrice", "Country"]
df_raw = pd.read_excel(DATA_PATH, usecols=usecols)

print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
display(df_raw.head())
display(df_raw.sample(5, random_state=42))


Shape: 541,909 rows x 6 columns


,InvoiceNo,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,6,2010-12-01 08:26:00,2.550,"17,850.000",United Kingdom
1,536365,6,2010-12-01 08:26:00,3.390,"17,850.000",United Kingdom
2,536365,8,2010-12-01 08:26:00,2.750,"17,850.000",United Kingdom
3,536365,6,2010-12-01 08:26:00,3.390,"17,850.000",United Kingdom
4,536365,6,2010-12-01 08:26:00,3.390,"17,850.000",United Kingdom


,InvoiceNo,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
209268,555200,24,2011-06-01 12:05:00,0.850,"17,315.000",United Kingdom
207108,554974,4,2011-05-27 17:14:00,6.950,"14,031.000",United Kingdom
167085,550972,4,2011-04-21 17:05:00,0.650,"14,031.000",United Kingdom
471836,576652,3,2011-11-16 10:39:00,1.950,"17,198.000",United Kingdom
115865,546157,2,2011-03-10 08:40:00,9.950,"13,502.000",United Kingdom


## Validation And Cleaning

Retention should be based on real completed purchase behavior. Remove rows with missing customers, cancellations, returns, and non-positive order economics before building cohorts.


In [3]:
validation_report = pd.DataFrame(
    {
        "metric": [
            "rows",
            "missing customer ids",
            "missing invoice dates",
            "cancel invoices",
            "non-positive quantity",
            "non-positive unit price",
        ],
        "value": [
            len(df_raw),
            int(df_raw["CustomerID"].isna().sum()),
            int(df_raw["InvoiceDate"].isna().sum()),
            int(df_raw["InvoiceNo"].astype(str).str.startswith("C").sum()),
            int((df_raw["Quantity"] <= 0).sum()),
            int((df_raw["UnitPrice"] <= 0).sum()),
        ],
    }
)

df = df_raw.copy()
df = df.dropna(subset=["CustomerID", "InvoiceDate"])
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()

df["CustomerID"] = df["CustomerID"].astype(int)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["OrderValue"] = df["Quantity"] * df["UnitPrice"]
df["InvoiceMonth"] = df["InvoiceDate"].dt.to_period("M")
df["InvoiceMonthStart"] = df["InvoiceMonth"].dt.to_timestamp()

display(validation_report)
print(f"Cleaned rows: {len(df):,}")
print(f"Customers retained for analysis: {df['CustomerID'].nunique():,}")
print(f"Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")


,metric,value
0,rows,541909
1,missing customer ids,135080
2,missing invoice dates,0
3,cancel invoices,9288
4,non-positive quantity,10624
5,non-positive unit price,2517


Cleaned rows: 397,884
Customers retained for analysis: 4,338
Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


## Build Customer Cohorts And Segment Proxies

The first purchase month becomes the cohort month. Customer type is assigned from first-order value terciles, and source is approximated using domestic versus international market location.


In [4]:
first_purchase = (
    df.groupby("CustomerID", as_index=False)
    .agg(
        CohortMonthStart=("InvoiceMonthStart", "min"),
        FirstInvoiceDate=("InvoiceDate", "min"),
    )
)
first_purchase["CohortMonth"] = first_purchase["CohortMonthStart"].dt.to_period("M")

first_orders = (
    df.sort_values(["CustomerID", "InvoiceDate", "InvoiceNo"])
    .groupby("CustomerID", as_index=False)
    .first()[["CustomerID", "InvoiceNo", "Country"]]
)

first_order_values = (
    df.merge(first_orders[["CustomerID", "InvoiceNo"]], on=["CustomerID", "InvoiceNo"], how="inner")
    .groupby("CustomerID", as_index=False)
    .agg(FirstOrderValue=("OrderValue", "sum"))
)

customer_dim = first_purchase.merge(first_orders, on="CustomerID", how="left").merge(first_order_values, on="CustomerID", how="left")
customer_dim["SourceProxy"] = np.where(
    customer_dim["Country"].eq("United Kingdom"),
    "Domestic (UK)",
    "International",
)
customer_dim["CustomerType"] = pd.qcut(
    customer_dim["FirstOrderValue"],
    q=3,
    labels=["Entry", "Core", "Premium"],
    duplicates="drop",
)
customer_dim["CustomerType"] = customer_dim["CustomerType"].astype(str)

activity = (
    df[["CustomerID", "InvoiceMonthStart"]]
    .drop_duplicates()
    .merge(customer_dim[["CustomerID", "CohortMonthStart", "CohortMonth", "SourceProxy", "CustomerType"]], on="CustomerID", how="left")
)
activity["MonthIndex"] = (
    (activity["InvoiceMonthStart"].dt.year - activity["CohortMonthStart"].dt.year) * 12
    + (activity["InvoiceMonthStart"].dt.month - activity["CohortMonthStart"].dt.month)
)

display(customer_dim.head())
display(activity.head())


,CustomerID,CohortMonthStart,FirstInvoiceDate,CohortMonth,InvoiceNo,Country,FirstOrderValue,SourceProxy,CustomerType
0,12346,2011-01-01,2011-01-18 10:01:00,2011-01,541431,United Kingdom,"77,183.600",Domestic (UK),Premium
1,12347,2010-12-01,2010-12-07 14:57:00,2010-12,537626,Iceland,711.790,International,Premium
2,12348,2010-12-01,2010-12-16 19:09:00,2010-12,539318,Finland,892.800,International,Premium
3,12349,2011-11-01,2011-11-21 09:51:00,2011-11,577609,Italy,"1,757.550",International,Premium
4,12350,2011-02-01,2011-02-02 16:01:00,2011-02,543037,Norway,334.400,International,Core


,CustomerID,InvoiceMonthStart,CohortMonthStart,CohortMonth,SourceProxy,CustomerType,MonthIndex
0,17850,2010-12-01,2010-12-01,2010-12,Domestic (UK),Entry,0
1,13047,2010-12-01,2010-12-01,2010-12,Domestic (UK),Core,0
2,12583,2010-12-01,2010-12-01,2010-12,International,Premium,0
3,13748,2010-12-01,2010-12-01,2010-12,Domestic (UK),Entry,0
4,15100,2010-12-01,2010-12-01,2010-12,Domestic (UK),Core,0


## Cohort Retention Matrix

This heatmap shows the percent of each signup-month cohort that returns in subsequent months. Month `0` is always `100%` because that is the acquisition month by definition.


In [5]:
cohort_counts = (
    activity.groupby(["CohortMonth", "MonthIndex"], observed=True)["CustomerID"]
    .nunique()
    .reset_index(name="ActiveCustomers")
)
cohort_sizes = cohort_counts[cohort_counts["MonthIndex"] == 0][["CohortMonth", "ActiveCustomers"]].rename(columns={"ActiveCustomers": "CohortSize"})
cohort_retention = cohort_counts.merge(cohort_sizes, on="CohortMonth", how="left")
cohort_retention["RetentionRate"] = cohort_retention["ActiveCustomers"] / cohort_retention["CohortSize"]

retention_matrix = cohort_retention.pivot(index="CohortMonth", columns="MonthIndex", values="RetentionRate").sort_index()
retention_matrix.index = retention_matrix.index.astype(str)

display(cohort_retention.head(12).round(3))

plt.figure(figsize=(14, 7))
sns.heatmap(retention_matrix, annot=True, fmt=".0%", cmap="YlGnBu", vmin=0, vmax=1)
plt.title("Monthly Cohort Retention Matrix")
plt.xlabel("Months Since Signup Month")
plt.ylabel("Signup Month Cohort")
plt.show()


,CohortMonth,MonthIndex,ActiveCustomers,CohortSize,RetentionRate
0,2010-12,0,885,885,1.000
1,2010-12,1,324,885,0.366
2,2010-12,2,286,885,0.323
3,2010-12,3,340,885,0.384
4,2010-12,4,321,885,0.363
5,2010-12,5,352,885,0.398
6,2010-12,6,321,885,0.363
7,2010-12,7,309,885,0.349
8,2010-12,8,313,885,0.354
9,2010-12,9,350,885,0.395


C:\Users\ahmad\AppData\Local\Temp\ipykernel_32804\3015240198.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Retention Milestones By Signup Month

Month 1 shows early habit formation, month 3 shows whether repeat behavior survives initial novelty, and month 6 captures durable purchase retention.


In [6]:
milestone_months = [1, 3, 6]
milestone_summary = cohort_retention[cohort_retention["MonthIndex"].isin(milestone_months)].pivot(
    index="CohortMonth", columns="MonthIndex", values="RetentionRate"
)
milestone_summary = milestone_summary.rename(columns={1: "Month1", 3: "Month3", 6: "Month6"}).sort_index()
milestone_summary.index = milestone_summary.index.astype(str)

display(milestone_summary.round(3))

milestone_plot = milestone_summary.reset_index().melt(id_vars="CohortMonth", var_name="Milestone", value_name="RetentionRate")
plt.figure(figsize=(14, 6))
sns.lineplot(data=milestone_plot, x="CohortMonth", y="RetentionRate", hue="Milestone", marker="o")
plt.title("Month 1, 3, And 6 Retention By Signup Cohort")
plt.xlabel("Signup month cohort")
plt.ylabel("Retention rate")
plt.xticks(rotation=45)
plt.show()


MonthIndex,Month1,Month3,Month6
CohortMonth,,,
2010-12,0.366,0.384,0.363
2011-01,0.221,0.230,0.247
2011-02,0.187,0.284,0.253
2011-03,0.150,0.199,0.268
2011-04,0.213,0.210,0.217
2011-05,0.190,0.173,0.264
2011-06,0.174,0.264,0.095
2011-07,0.181,0.223,NaN
2011-08,0.207,0.243,NaN


C:\Users\ahmad\AppData\Local\Temp\ipykernel_32804\927947741.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Source Proxy Comparison

The dataset has no explicit acquisition channel, so `Country` is used as a market/source proxy. This answers whether domestic and international customers retain differently after their first observed purchase.


In [7]:
source_counts = (
    activity.groupby(["SourceProxy", "CohortMonth", "MonthIndex"], observed=True)["CustomerID"]
    .nunique()
    .reset_index(name="ActiveCustomers")
)
source_sizes = source_counts[source_counts["MonthIndex"] == 0][["SourceProxy", "CohortMonth", "ActiveCustomers"]].rename(columns={"ActiveCustomers": "CohortSize"})
source_retention = source_counts.merge(source_sizes, on=["SourceProxy", "CohortMonth"], how="left")
source_retention["RetentionRate"] = source_retention["ActiveCustomers"] / source_retention["CohortSize"]

avg_source_curve = (
    source_retention.groupby(["SourceProxy", "MonthIndex"], observed=True)["RetentionRate"]
    .mean()
    .reset_index()
)
source_milestones = source_retention[source_retention["MonthIndex"].isin([1, 3, 6])].groupby(["SourceProxy", "MonthIndex"], observed=True)["RetentionRate"].mean().unstack()
source_milestones = source_milestones.rename(columns={1: "Month1", 3: "Month3", 6: "Month6"})

display(source_milestones.round(3))

plt.figure(figsize=(12, 6))
sns.lineplot(data=avg_source_curve, x="MonthIndex", y="RetentionRate", hue="SourceProxy", marker="o")
plt.title("Average Retention Curve By Source Proxy")
plt.xlabel("Months since signup month")
plt.ylabel("Retention rate")
plt.xlim(left=0)
plt.show()


MonthIndex,Month1,Month3,Month6
SourceProxy,,,
Domestic (UK),0.206,0.231,0.245
International,0.213,0.247,0.239


C:\Users\ahmad\AppData\Local\Temp\ipykernel_32804\1406802610.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Customer Type Comparison

Customer type is based on first-order value terciles. This tests whether stronger first-order economics are associated with stronger repeat behavior over time.


In [8]:
type_counts = (
    activity.groupby(["CustomerType", "CohortMonth", "MonthIndex"], observed=True)["CustomerID"]
    .nunique()
    .reset_index(name="ActiveCustomers")
)
type_sizes = type_counts[type_counts["MonthIndex"] == 0][["CustomerType", "CohortMonth", "ActiveCustomers"]].rename(columns={"ActiveCustomers": "CohortSize"})
type_retention = type_counts.merge(type_sizes, on=["CustomerType", "CohortMonth"], how="left")
type_retention["RetentionRate"] = type_retention["ActiveCustomers"] / type_retention["CohortSize"]

avg_type_curve = (
    type_retention.groupby(["CustomerType", "MonthIndex"], observed=True)["RetentionRate"]
    .mean()
    .reset_index()
)
type_milestones = type_retention[type_retention["MonthIndex"].isin([1, 3, 6])].groupby(["CustomerType", "MonthIndex"], observed=True)["RetentionRate"].mean().unstack()
type_milestones = type_milestones.rename(columns={1: "Month1", 3: "Month3", 6: "Month6"})

display(type_milestones.round(3))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.lineplot(data=avg_type_curve, x="MonthIndex", y="RetentionRate", hue="CustomerType", marker="o", ax=axes[0])
axes[0].set_title("Average Retention Curve By Customer Type")
axes[0].set_xlabel("Months since signup month")
axes[0].set_ylabel("Retention rate")

type_milestone_plot = type_milestones.reset_index().melt(id_vars="CustomerType", var_name="Milestone", value_name="RetentionRate")
sns.barplot(data=type_milestone_plot, x="CustomerType", y="RetentionRate", hue="Milestone", ax=axes[1])
axes[1].set_title("Retention Milestones By Customer Type")
axes[1].set_xlabel("")
axes[1].set_ylabel("Retention rate")

plt.tight_layout()
plt.show()


MonthIndex,Month1,Month3,Month6
CustomerType,,,
Core,0.209,0.235,0.243
Entry,0.171,0.185,0.184
Premium,0.239,0.279,0.305


C:\Users\ahmad\AppData\Local\Temp\ipykernel_32804\4113636130.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Strategy Lens: Best And Weakest Cohorts

Looking only at average retention can hide large differences across signup cohorts. This section identifies which cohorts overperformed and which lost momentum fastest.


In [9]:
best_m1 = milestone_summary["Month1"].dropna().sort_values(ascending=False)
best_m3 = milestone_summary["Month3"].dropna().sort_values(ascending=False)
best_m6 = milestone_summary["Month6"].dropna().sort_values(ascending=False)

cohort_rankings = pd.DataFrame(
    {
        "BestMonth1Cohort": [best_m1.index[0] if not best_m1.empty else None],
        "WorstMonth1Cohort": [best_m1.index[-1] if not best_m1.empty else None],
        "BestMonth3Cohort": [best_m3.index[0] if not best_m3.empty else None],
        "WorstMonth3Cohort": [best_m3.index[-1] if not best_m3.empty else None],
        "BestMonth6Cohort": [best_m6.index[0] if not best_m6.empty else None],
        "WorstMonth6Cohort": [best_m6.index[-1] if not best_m6.empty else None],
    }
)

display(cohort_rankings)

cohort_size_overview = cohort_sizes.copy()
cohort_size_overview["CohortMonth"] = cohort_size_overview["CohortMonth"].astype(str)
plt.figure(figsize=(13, 5))
sns.barplot(data=cohort_size_overview, x="CohortMonth", y="CohortSize", color="#2a9d8f")
plt.title("Cohort Size By Signup Month")
plt.xlabel("Signup month cohort")
plt.ylabel("Customers in cohort")
plt.xticks(rotation=45)
plt.show()


,BestMonth1Cohort,WorstMonth1Cohort,BestMonth3Cohort,WorstMonth3Cohort,BestMonth6Cohort,WorstMonth6Cohort
0,2010-12,2011-11,2010-12,2011-09,2010-12,2011-06


C:\Users\ahmad\AppData\Local\Temp\ipykernel_32804\547856752.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Statistical Checks

These tests do not replace business judgment, but they help confirm whether early-value customer types and source proxies are associated with materially different retention outcomes.


In [10]:
eligible_type_groups = []
for _, group in type_retention[type_retention["MonthIndex"] == 3].groupby("CustomerType", observed=True):
    values = group["RetentionRate"].dropna().values
    if len(values) > 1:
        eligible_type_groups.append(values)

if len(eligible_type_groups) >= 2:
    type_h, type_p = stats.kruskal(*eligible_type_groups)
else:
    type_h, type_p = np.nan, np.nan

customer_month3 = activity[activity["MonthIndex"] == 3][["CustomerID"]].drop_duplicates().assign(RetainedMonth3=1)
customer_stats = customer_dim.merge(customer_month3, on="CustomerID", how="left")
customer_stats["RetainedMonth3"] = customer_stats["RetainedMonth3"].fillna(0)
rho, rho_p = stats.spearmanr(customer_stats["FirstOrderValue"], customer_stats["RetainedMonth3"], nan_policy="omit")

stats_summary = pd.DataFrame(
    {
        "test": [
            "Kruskal-Wallis: Month 3 retention by customer type",
            "Spearman: First order value vs Month 3 retention",
        ],
        "statistic": [type_h, rho],
        "p_value": [type_p, rho_p],
    }
)

display(stats_summary.round(4))


,test,statistic,p_value
0,Kruskal-Wallis: Month 3 retention by customer ...,5.946,0.051
1,Spearman: First order value vs Month 3 retention,0.099,0.000


## Key Findings

Summarize the retention differences that matter most for growth strategy, onboarding design, and channel investment.


In [11]:
best_cohort_m1 = best_m1.index[0] if not best_m1.empty else "N/A"
best_cohort_m3 = best_m3.index[0] if not best_m3.empty else "N/A"
best_cohort_m6 = best_m6.index[0] if not best_m6.empty else "N/A"

source_m1 = source_milestones.loc[:, "Month1"].sort_values(ascending=False)
type_m3 = type_milestones.loc[:, "Month3"].sort_values(ascending=False)
type_m6 = type_milestones.loc[:, "Month6"].sort_values(ascending=False) if "Month6" in type_milestones.columns else pd.Series(dtype=float)

findings = []
findings.append(
    f"- Strongest early retention cohort: {best_cohort_m1} at {best_m1.iloc[0]:.1%} Month 1 retention." if not best_m1.empty else "- Month 1 retention cohorts were unavailable."
)
findings.append(
    f"- Strongest durable cohort: {best_cohort_m6} at {best_m6.iloc[0]:.1%} Month 6 retention." if not best_m6.empty else "- Month 6 retention coverage is limited because later cohorts have not fully matured."
)
if not source_m1.empty:
    findings.append(
        f"- Higher-retaining source proxy at Month 1: {source_m1.index[0]} ({source_m1.iloc[0]:.1%})."
    )
if not type_m3.empty:
    findings.append(
        f"- Best customer type at Month 3: {type_m3.index[0]} ({type_m3.iloc[0]:.1%})."
    )
if not type_m6.empty:
    findings.append(
        f"- Best customer type at Month 6: {type_m6.index[0]} ({type_m6.iloc[0]:.1%})."
    )
if pd.notna(rho):
    findings.append(
        f"- First-order value and Month 3 retention have Spearman correlation {rho:.3f}; higher-value acquisition tends to be associated with stronger repeat behavior." 
    )
if pd.notna(type_p):
    findings.append(
        f"- Customer-type retention differences at Month 3 produce p-value {type_p:.4f}, which helps gauge whether initial-value tiers behave materially differently."
    )

print("\n".join(findings))


- Strongest early retention cohort: 2010-12 at 36.6% Month 1 retention.
- Strongest durable cohort: 2010-12 at 36.3% Month 6 retention.
- Higher-retaining source proxy at Month 1: International (21.3%).
- Best customer type at Month 3: Premium (27.9%).
- Best customer type at Month 6: Premium (30.5%).
- First-order value and Month 3 retention have Spearman correlation 0.099; higher-value acquisition tends to be associated with stronger repeat behavior.
- Customer-type retention differences at Month 3 produce p-value 0.0512, which helps gauge whether initial-value tiers behave materially differently.


## Product And Marketing Implications

Product strategy:

- If Month 1 retention is weak across multiple cohorts, focus on first-order to second-order activation, replenishment reminders, and post-purchase onboarding.
- If premium first-order customers retain better, product teams should design premium follow-up experiences and merchandising that preserve that momentum.
- If later cohorts decay faster than earlier ones, review any pricing, assortment, or fulfillment changes that landed after acquisition.

Marketing strategy:

- If domestic cohorts outperform international ones, international acquisition may need better localization or shipping-value messaging before scaling spend.
- If premium cohorts retain materially better, optimize campaigns for higher-quality acquisition even at slightly higher CAC.
- If a specific signup month cohort overperforms, inspect the campaign mix or promotional environment active in that month and test it deliberately.

Interpretation guardrail:

- This notebook uses `Country` as a source proxy and first-order value as a customer type proxy because the source data has no explicit acquisition channel or account-plan field.
